In [ ]:
using UnfoldSim
using Random
using DataFrames
using CairoMakie
using StableRNGs
using UnfoldMakie
using Plots
using StatsBase

In [ ]:
# Simulate data for 4 conditions and 6 continuous levels

In [3]:
using UnfoldSim, Random, StatsModels, DataFrames, CSV, DSP, UnfoldMakie, Statistics

# Set seed for reproducibility
rng = MersenneTwister(1)
sfreq = 200  # sampling frequency

p1 = (p100(; sfreq=sfreq), @formula(0 ~ 1), [5.0], Dict(), 0)

n1 = (n170(; sfreq=sfreq),
      @formula(0 ~ 1 + condition),
[     5.0, 2.0, -2.0, 4.0],      Dict(), 0)

p3 = (p300(; sfreq=sfreq),
      @formula(0 ~ 1 + continuous),
      [5.0, 1.0],
      Dict(), 0)

n4 = (n400(; sfreq=sfreq),
      @formula(0 ~ 1 + condition),
      [5.0, 4.0, -4.0, 6.0],   # 4 weights
      Dict(), 0)

components = [p1, n1, p3, n4]

# Define Experimental Design
design = SingleSubjectDesign(
    conditions = Dict(
        :condition => ["car", "face", "house", "animal"],  # 4 conditions
        :continuous => range(-3, 3, length = 6),           # 6 levels
    ),
    event_order_function = shuffle,
) |> x -> RepeatDesign(x, 50)

# Spatial Structure for 32 channels
hart = UnfoldSim.headmodel()
l32, _ = UnfoldMakie.example_montage("biosemi_32")
full_labels = hart.electrodes["label"]
ordered_indices = [findfirst(isequal(ch), full_labels) for ch in l32]

# Run Simulation
data_SS, events_SS = UnfoldSim.predef_eeg(
    rng, design, LinearModelComponent, components;
    sfreq = sfreq,
    multichannel = ["Right Occipital Pole", "Left Postcentral Gyrus",
                    "Left Superior Frontal Gyrus", "Right Subcallosal Cortex"],
    return_epoched = true,
    noiselevel = 1
)

# Extract 32 channels
data_32_ch = data_SS[ordered_indices, :, :]

# Export to Long Format CSV
n_chan, n_time, n_trials = size(data_32_ch)

# Time vector (ms)
time_ms = collect(0:(n_time - 1)) .* (1000 / sfreq)
channel_names = Symbol.(l32)

df_long = DataFrame()
println("Exporting $n_trials trials...")

for t in 1:n_trials
    trial_data = permutedims(data_32_ch[:, :, t], (2, 1))
    temp_df = DataFrame(trial_data, channel_names)
    
    temp_df[!, :Time_ms]    = time_ms
    temp_df[!, :Trial_ID]   .= t
    temp_df[!, :Condition]  .= events_SS.condition[t]
    temp_df[!, :Continuous] .= events_SS.continuous[t]
    
    append!(df_long, temp_df)
end

CSV.write("4conditions_noise1_.csv", df_long)

println("CSV rows=$(nrow(df_long)), cols=$(ncol(df_long)), trials=$(n_trials), timepoints=$(n_time), channels=$(n_chan)")
println("Done! Saved csv")

Please cite: HArtMuT: Harmening Nils, Klug Marius, Gramann Klaus and Miklody Daniel - 10.1088/1741-2552/aca8ce
Please cite: HArtMuT: Harmening Nils, Klug Marius, Gramann Klaus and Miklody Daniel - 10.1088/1741-2552/aca8ce
Exporting 1200 trials...
CSV rows=144000, cols=36, trials=1200, timepoints=120, channels=32
Done! Saved to DR_case_noise4.csv


In [ ]:
#topoplot visualisation

In [ ]:
using CairoMakie
l32, montage = UnfoldMakie.example_montage("biosemi_32")
positions = montage
# Average across trials
mean_data = mean(data_32_ch, dims=3)[:, :, 1]

# Convert to dataframe
df_topo = UnfoldMakie.eeg_array_to_dataframe(
    mean_data,
    string.(l32)
)

# Fix time axis (ms)
unique_times = sort(unique(df_topo.time))
time_mapping = Dict(zip(unique_times, time_ms))

df_topo.time = [time_mapping[t] for t in df_topo.time]

# Plot
f = Figure(size = (1200, 700))

plot_topoplotseries!(
    f[1, 1],
    df_topo;
    bin_num = 16,
    nrows = 4,
    positions = positions,

    visual = (
        label_scatter = false,
        contours = false,
    ),

    axis = (
        xlabel = "Time [ms]",
    ),
)

f

In [ ]:
# Example for simulating data for 10 conditions and 2 continuous levels

In [ ]:
using UnfoldSim, Random, StatsModels, DataFrames, CSV, DSP, UnfoldMakie, Statistics

rng = MersenneTwister(1)
sfreq = 200

# Define Components
p1 = (p100(; sfreq=sfreq), @formula(0 ~ 1),
      [5.0],
      Dict(), 0)

# N170: each condition has a different amplitude
# reference = car, then bird, chair, dog, face, house, phone, shoe, table, tree
n1 = (n170(; sfreq=sfreq), @formula(0 ~ 1 + condition),
      [5.0, 3.0, -2.0, 2.0, 5.0, -1.0, -3.0, 1.0, -2.0, 0.0],
      Dict(), 0)

# P300: continuous variable (0 or 1)
p3 = (p300(; sfreq=sfreq), @formula(0 ~ 1 + continuous),
      [5.0, 2.0],
      Dict(), 0)

n4 = (n400(; sfreq=sfreq), @formula(0 ~ 1 + condition),
      [5.0, 4.0, -3.0, 4.0, 3.0, -2.0, -3.0, -2.0, -3.0, 3.0],
      Dict(), 0)

components = [p1, n1, p3, n4]

# 10 conditions x 2 continuous levels x 100 repeats = 2000 trials
design = SingleSubjectDesign(
    conditions = Dict(
        :condition  => ["car", "bird", "chair", "dog", "face",
                        "house", "phone", "shoe", "table", "tree"],
        :continuous => [0.0, 1.0],
    ),
    event_order_function = shuffle,
) |> x -> RepeatDesign(x, 100)

# 32 channels filter
hart = UnfoldSim.headmodel()
l32, _ = UnfoldMakie.example_montage("biosemi_32")
full_labels = hart.electrodes["label"]
ordered_indices = [findfirst(isequal(ch), full_labels) for ch in l32]


data_SS, events_SS = UnfoldSim.predef_eeg(
    rng, design, LinearModelComponent, components;
    sfreq = sfreq,
    multichannel = ["Right Occipital Pole", "Left Postcentral Gyrus",
                    "Left Superior Frontal Gyrus", "Right Subcallosal Cortex"],
    return_epoched = true,
    noiselevel = 2
)

# Extract 32 channels
data_32_ch = data_SS[ordered_indices, :, :]

#Export to Long Format CSV
n_chan, n_time, n_trials = size(data_32_ch)
time_ms = collect(0:(n_time - 1)) .* (1000 / sfreq)
channel_names = Symbol.(l32)

df_long = DataFrame()
println("Exporting $n_trials trials...")

for t in 1:n_trials
    trial_data = permutedims(data_32_ch[:, :, t], (2, 1))
    temp_df = DataFrame(trial_data, channel_names)

    temp_df[!, :Time_ms]    = time_ms
    temp_df[!, :Trial_ID]   .= t
    temp_df[!, :Condition]  .= events_SS.condition[t]
    temp_df[!, :Continuous] .= events_SS.continuous[t]

    append!(df_long, temp_df)
end

CSV.write("7may_DR_EEG_10conditions_noise2.csv", df_long)
println("Done! Saved to DR_EEG_10conditions.csv")
println("Trials: $n_trials  |  Time: $n_time  |  Channels: $n_chan")
println("Done! Saved csv")